## Preprocessing of Midi Files

Goals:
- read MIDI files
- align with CSV file (train, test, validation split)
- filter non-4/4
- filter fills
- extract Information: 
    - onsets (hits) (binary)
    - continuous derivation (-.5, .5)
    - velocity (0 - 127 --> 0 - 1)

TODOS
- train, test, validation split
- remove duplicates if 2 hits on same instrument are quantized to same timestep --> use louder one


Problems:
- The original paper (2019) summarizes into 9 categories that differ from the second paper (where we derive our dataset), which has 7 categories; For now I use the first mapping with 9 instruments. 

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
import pretty_midi
import time
from tqdm.notebook import tqdm

# ! python -m pip install ipywidgets

In [7]:
# Constants

# DS_ROOT = "e-gmd-v1.0.0"
DS_ROOT = "../../e-gmd-v1.0.0" # this assumes the dataset to be in a solder next to the git repository
N_INSTRUMENTS = 9

In [8]:
def pitch_to_category(pitches):
    """
    Maps the Midi pitch values 0 - 127 to the Drum category according to the Paper's Appendix B
    """

    mapping = {
        36 : 36, # Kick
        38 : 38, # Snare Head
        40 : 38, # Snare Rim
        37 : 38, # Snare Side Stick
        48 : 50, # Tom High
        50 : 50,
        45 : 47,
        47 : 47,
        43 : 43,
        58 : 43,
        46 : 46,
        26 : 46,
        42 : 42,
        22 : 42,
        44 : 42,
        49 : 49,
        55 : 49,
        57 : 49,
        52 : 49,
        51 : 51,
        59 : 51,
        53 : 51,
        # I added this according to the 2020 Paper where Tambourine (54) is counted as HiHat
        54 : 42,
        39 : 38, # 'Hand Clap' --> Snare (I assume)
        56 : 51, # Cowbell --> Ride

    }

    # how many instruments?
    n_instruments = len(set(mapping.values()))
    if n_instruments != N_INSTRUMENTS:
        raise ValueError("Global constant N_INSTRUMENTS does not fit unique values in mapping dict")
    
    
    categories = []
    for pitch in pitches:
        # write 0 if value is missing in mapping
        categories.append(mapping.get(pitch, 0))
        if categories[-1] == 0:
            print(f"{pitch} not in mapping")
    return np.array(categories)


In [ ]:
# Read MIDI file, extract all needed data

def read_midi(midi_path : Path, tempo_bpm):

    try:
        midi_data = pretty_midi.PrettyMIDI(midi_path)
    except Exception as e:
        raise ValueError(f"Failed to parse MIDI file {midi_path}: {e}")
    
    # Get drum track (should be only drum track)
    drum_track = None
    for instrument in midi_data.instruments:
        if instrument.is_drum:
            drum_track = instrument
            break

    if drum_track is None:
        raise ValueError(f"No drum track found in {midi_path}")
    
    # iterate over notes
    onsets = []
    pitches = []
    velocities = []
    for note in drum_track.notes:
        onsets.append(note.start)
        pitches.append(note.pitch)
        velocities.append(note.velocity)



    # sort by onset time
    sort_idx = np.argsort(onsets)
    onsets = np.array(onsets)[sort_idx]
    pitches = np.array(pitches)[sort_idx]
    velocities = np.array(velocities)[sort_idx] / 126.


    # calc 1/16th notes per second
    bps = tempo_bpm / 60. # beats per second
    sixteenth_ps = bps * 4 # sixteenth notes per second
    step = 1.0 / sixteenth_ps

    # get vector / grid length for binary Matrix
    last_onset = onsets[-1] # in sec
    n_grid_onsets = int(last_onset*sixteenth_ps) + 2 # TODO should only be +1; bug?

    # snap to grid
    nearest_idc = np.rint(onsets * sixteenth_ps).astype(np.int32) # get snapped onsets as grid indices
    nearest = nearest_idc * step # snapped values

    
    offsets = np.abs(onsets - nearest) # timing derivations ('feel')
    offset_relative = offsets * sixteenth_ps # relative to grid, not in seconds


    # create instrument grouped array --> "round" different drum parts (ride bell, ride edge --> ride)
    grouped_pitches = pitch_to_category(pitches)
    pitch_categories = np.array([36, 38, 50, 47, 43, 46, 42, 49, 51])
    pitch_to_row = {p: i for i, p in enumerate(pitch_categories)}
    pitch_rows = np.array([pitch_to_row[p] for p in grouped_pitches])

    
    # create onset binary grid
    onset_grid = np.zeros((len(pitch_categories), n_grid_onsets), dtype=np.uint8)
    onset_grid[pitch_rows, nearest_idc] = 1
    
    # create offset (deviation) grid
    offset_grid = np.zeros_like(onset_grid, dtype=np.float32)
    offset_grid[pitch_rows, nearest_idc] = offset_relative

    # create velocity grid
    vel_grid = np.zeros_like(onset_grid, dtype=np.float32)
    vel_grid[pitch_rows, nearest_idc] = velocities
    
    return onset_grid, offset_grid, vel_grid

In [10]:
# Iterate over csv
# read all MIDI files

csv_path = Path(DS_ROOT) / "e-gmd-v1.0.0.csv"
df = pd.read_csv(csv_path)

# filter all non 4-4
df_filt = df[df['time_signature'] == '4-4']
filter_count = len(df) - len(df_filt)

start = time.perf_counter()
filter_count = 0
for row in tqdm(df_filt.itertuples(index=False), total=len(df_filt)):

    # early stopping for testing
    # if i == 2:
    #     break
    
    # get BPM from csv
    bpm = row.bpm

    # load file
    filepath =  Path(DS_ROOT) / row.midi_filename
    # print(f"Reading {filepath} ...")

    # call reader function
    onset_grid, offset_grid, vel_grid = read_midi(filepath, bpm)
    
elapsed = time.perf_counter() - start
print(f"Processed {len(df)} MIDI files in {elapsed:.6f} seconds")
print(f"Removed {filter_count} files due to mismatching time signature")


  0%|          | 0/45021 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Questions

- do i concatenate the 2-bar snippets (sliding window of 1 bar) fully, step-by-step or not at all?

In [ ]:
len(df)

45537